# Paper Figures

Generate publication-ready figures for the thesis.

**Part 1 (now):** Data-dependent figures  
**Part 2 (later):** Model-dependent figures — run after final model is ready

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Publication style
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Saving figures to: {FIGURES_DIR.resolve()}")

In [ ]:
# Load data
BASE_DIR = Path("..")
df_train = pd.read_csv(BASE_DIR / "data/processed/train.csv")
df_test = pd.read_csv(BASE_DIR / "data/processed/test.csv")

mapping = pd.read_csv(BASE_DIR / "data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

df_train["supercategory"] = df_train["label"].map(label_to_supercat)
df_test["supercategory"] = df_test["label"].map(label_to_supercat)

print(f"Train: {len(df_train)}, Test: {len(df_test)}")

---
## Part 1: Data-dependent figures (run now)

In [ ]:
# Figure 1: City Distribution
city_counts = df_train["city_group"].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))

# Top 15 cities + Other aggregated
top_n = 15
top_cities = city_counts.head(top_n)
other_count = city_counts.iloc[top_n:].sum()

plot_data = pd.concat([top_cities, pd.Series({"Other (small)": other_count})])

colors = ['#2ecc71' if i < 3 else '#3498db' if i < 10 else '#95a5a6' 
          for i in range(len(plot_data))]

bars = ax.bar(range(len(plot_data)), plot_data.values, color=colors, edgecolor='white')
ax.set_xticks(range(len(plot_data)))
ax.set_xticklabels(plot_data.index, rotation=45, ha='right')
ax.set_ylabel('Number of samples')
ax.set_xlabel('City group')
ax.set_title('Distribution of resumes across city groups (training set)')

# Add value labels
for bar, val in zip(bars, plot_data.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            f'{val:,}', ha='center', va='bottom', fontsize=8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "city_distribution.pdf")
plt.savefig(FIGURES_DIR / "city_distribution.png")
plt.show()
print(f"Saved: city_distribution.pdf")

In [ ]:
# Figure 2: Class Distribution
class_counts = df_train["supercategory"].value_counts()

# Shorter names for plot
short_names = {
    'backend_general_dev': 'Backend Dev',
    'web_frontend': 'Frontend',
    'sysadmin_devops_network': 'SysAdmin/DevOps',
    'project_product': 'PM/Product',
    'sales_account': 'Sales',
    'tech_support_helpdesk': 'Tech Support',
    'it_governance_leadership': 'IT Leadership',
    'technical_specialized': 'Tech Specialist',
    'generic_it_ops': 'IT Operations'
}

fig, ax = plt.subplots(figsize=(10, 5))

class_data = class_counts.rename(index=short_names)
colors = plt.cm.Set3(np.linspace(0, 1, len(class_data)))

bars = ax.barh(range(len(class_data)), class_data.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(class_data)))
ax.set_yticklabels(class_data.index)
ax.set_xlabel('Number of samples')
ax.set_title('Distribution of professional categories (training set)')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, class_data.values):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2, 
            f'{val:,}', ha='left', va='center', fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.pdf")
plt.savefig(FIGURES_DIR / "class_distribution.png")
plt.show()
print(f"Saved: class_distribution.pdf")

In [ ]:
# Figure 3: City × Class heatmap (sample counts)
# Shows which city-class combinations have low support

top_cities_list = df_train["city_group"].value_counts().head(10).index.tolist()
df_subset = df_train[df_train["city_group"].isin(top_cities_list)]

pivot = pd.crosstab(df_subset["city_group"], df_subset["supercategory"])
pivot = pivot.loc[top_cities_list]  # Keep order
pivot.columns = [short_names.get(c, c) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Sample count'})

ax.set_xlabel('Professional category')
ax.set_ylabel('City group')
ax.set_title('Sample distribution: City × Category (top 10 cities)')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "city_class_heatmap.pdf")
plt.savefig(FIGURES_DIR / "city_class_heatmap.png")
plt.show()
print(f"Saved: city_class_heatmap.pdf")

In [ ]:
# Figure 4: Sample weight distribution (sqrt reweighting)
city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
norm_w = raw_w / raw_w.mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Weight by city
top_10 = norm_w.head(10)
bottom_10 = norm_w.tail(10)
weight_data = pd.concat([top_10, bottom_10]).drop_duplicates()

colors = ['#e74c3c' if w > 1 else '#3498db' for w in weight_data.values]
ax1.barh(range(len(weight_data)), weight_data.values, color=colors)
ax1.set_yticks(range(len(weight_data)))
ax1.set_yticklabels(weight_data.index)
ax1.axvline(x=1.0, color='black', linestyle='--', alpha=0.5, label='Mean weight')
ax1.set_xlabel('Sample weight')
ax1.set_title('Sqrt-inverse reweighting by city')
ax1.legend()

# Right: Weight distribution histogram
df_train["weight"] = df_train["city_group"].map(norm_w.to_dict())
ax2.hist(df_train["weight"], bins=30, color='#3498db', edgecolor='white', alpha=0.7)
ax2.axvline(x=1.0, color='red', linestyle='--', label='Mean = 1.0')
ax2.set_xlabel('Sample weight')
ax2.set_ylabel('Number of samples')
ax2.set_title('Distribution of sample weights')
ax2.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "reweighting_distribution.pdf")
plt.savefig(FIGURES_DIR / "reweighting_distribution.png")
plt.show()
print(f"Saved: reweighting_distribution.pdf")

---
## Part 2: Model-dependent figures (run AFTER final model)

**TODO:** Update `FINAL_MODEL` path and re-run these cells

In [ ]:
# === CONFIGURE THIS AFTER TRAINING ===
# Change to your final model path
FINAL_MODEL = "models/bert_9classes_final"  # or bert_oversample_only / bert_gdro_eta01_2ep

# Model results (fill in after training)
RESULTS = {
    "BERT baseline": {"accuracy": 0.592, "f1": 0.608, "tpr_worst": 0.329, "tpr_macro": 0.116},
    "+ sqrt_rw (2ep)": {"accuracy": 0.609, "f1": 0.621, "tpr_worst": 0.329, "tpr_macro": 0.116},
    # "+ oversample": {"accuracy": ???, "f1": ???, "tpr_worst": ???, "tpr_macro": ???},
    # "+ GroupDRO": {"accuracy": ???, "f1": ???, "tpr_worst": ???, "tpr_macro": ???},
}

print("Fill in RESULTS dict after models finish training")

In [ ]:
# Figure 5: Model Comparison (Accuracy vs Fairness trade-off)
# UNCOMMENT AND RUN AFTER FILLING RESULTS

'''
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

models = list(RESULTS.keys())
accuracies = [RESULTS[m]["accuracy"] for m in models]
tpr_worst = [RESULTS[m]["tpr_worst"] for m in models]
tpr_macro = [RESULTS[m]["tpr_macro"] for m in models]

x = np.arange(len(models))
width = 0.35

# Left: Accuracy comparison
bars1 = ax1.bar(x, accuracies, width, label='Accuracy', color='#3498db')
ax1.set_ylabel('Score')
ax1.set_title('Model Accuracy Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=15, ha='right')
ax1.set_ylim(0.4, 0.7)
ax1.axhline(y=0.609, color='green', linestyle='--', alpha=0.5, label='Best accuracy')
ax1.legend()

# Right: Fairness comparison
bars2 = ax2.bar(x - width/2, tpr_worst, width, label='TPR Gap (worst)', color='#e74c3c')
bars3 = ax2.bar(x + width/2, tpr_macro, width, label='TPR Gap (macro)', color='#f39c12')
ax2.set_ylabel('TPR Gap (lower is better)')
ax2.set_title('Fairness Comparison (robust metrics)')
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=15, ha='right')
ax2.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_comparison.pdf")
plt.savefig(FIGURES_DIR / "model_comparison.png")
plt.show()
'''

print("Uncomment and run after filling RESULTS dict")

In [ ]:
# Figure 6: TPR Heatmap (final model)
# UNCOMMENT AND RUN AFTER FINAL MODEL

'''
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
import joblib

# Load model
model_path = Path(FINAL_MODEL)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
le = joblib.load(model_path / "label_encoder.joblib")

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
model.eval()

# Get predictions
df_test["supercategory"] = df_test["label"].map(label_to_supercat)
df_test["y"] = le.transform(df_test["supercategory"])

predictions = []
with torch.no_grad():
    for text in df_test["resume_text"].values:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        pred = outputs.logits.argmax(-1).item()
        predictions.append(pred)

df_test["y_pred"] = predictions

# Compute TPR by city × class
top_cities = df_test["city_group"].value_counts().head(10).index.tolist()
df_subset = df_test[df_test["city_group"].isin(top_cities)]

tpr_matrix = []
for city in top_cities:
    row = []
    for c in range(len(le.classes_)):
        mask = (df_subset["city_group"] == city) & (df_subset["y"] == c)
        if mask.sum() >= 10:
            correct = ((df_subset[mask]["y_pred"] == c)).sum()
            tpr = correct / mask.sum()
        else:
            tpr = np.nan
        row.append(tpr)
    tpr_matrix.append(row)

tpr_df = pd.DataFrame(tpr_matrix, index=top_cities, 
                      columns=[short_names.get(c, c) for c in le.classes_])

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(tpr_df, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax,
            linewidths=0.5, vmin=0, vmax=1, cbar_kws={'label': 'True Positive Rate'})
ax.set_xlabel('Professional category')
ax.set_ylabel('City group')
ax.set_title('TPR by City × Category (final model)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "tpr_heatmap_final.pdf")
plt.savefig(FIGURES_DIR / "tpr_heatmap_final.png")
plt.show()
'''

print("Uncomment and run after final model is ready")

In [ ]:
# Summary: List all generated figures
print("\n" + "="*60)
print("GENERATED FIGURES")
print("="*60)
for f in sorted(FIGURES_DIR.glob("*.pdf")):
    print(f"  {f.name}")
print(f"\nLocation: {FIGURES_DIR.resolve()}")
print("\nUse in LaTeX:")
print("  \\includegraphics[width=\\linewidth]{figures/city_distribution.pdf}")